# FLAX MLP on Synthesized Spectra (`params` + `mu` -> `flux`)

This notebook demonstrates supervised training of a FLAX MLP using synthesized spectra stored in a Zarr dataset.

Pipeline:
1. Load data with `scripts/jax_spectra_dataloader.py`.
2. Build inputs from physical parameters plus limb angle `mu`.
3. Train a `flax.linen` MLP to predict flux spectra and track train/validation loss.
4. Visualize a held-out prediction.

## Environment Pin (Recommended)

Use the pinned dependency set in `requirements-flax-ml.txt` before running this notebook:

`pip install -r requirements-flax-ml.txt`

Pinned versions:
- `numpy==1.26.4`
- `jax==0.4.28`
- `jaxlib==0.4.28`
- `flax==0.8.1`
- `optax==0.1.9`

If you change versions, restart the kernel before running training cells.

In [ ]:
import os
from pathlib import Path
import sys
import importlib.metadata as md

os.environ.setdefault("JAX_PLATFORMS", "cpu")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp")

PINNED = {
    "numpy": "1.26.4",
    "jax": "0.4.28",
    "jaxlib": "0.4.28",
    "flax": "0.8.1",
    "optax": "0.1.9",
}

import numpy as np
import zarr
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

try:
    import flax.linen as nn
    from flax.training import train_state
    import optax
except Exception as exc:
    versions = {}
    for pkg in PINNED:
        try:
            versions[pkg] = md.version(pkg)
        except Exception:
            versions[pkg] = "not installed"
    raise RuntimeError(
        "Failed to import FLAX/Optax. Installed versions: "
        f"{versions}. Install the pinned stack with: pip install -r requirements-flax-ml.txt"
    ) from exc

installed = {}
for pkg in PINNED:
    try:
        installed[pkg] = md.version(pkg)
    except Exception:
        installed[pkg] = "not installed"

mismatches = {pkg: (installed[pkg], want) for pkg, want in PINNED.items() if installed[pkg] != want}
if mismatches:
    print("Version warning (non-pinned environment):", mismatches)
    print("Recommended: pip install -r requirements-flax-ml.txt")

repo_root = Path().resolve()
if not (repo_root / "scripts" / "jax_spectra_dataloader.py").exists():
    raise FileNotFoundError("Run this notebook from the Turbospectrum_NLTE repository root.")

sys.path.insert(0, str(repo_root))

from scripts.jax_spectra_dataloader import create_jax_spectra_dataloaders

# Fail early with a clear error if runtime math ops are broken.
try:
    _ = (jnp.asarray([1.0], dtype=jnp.float32) + 1.0).block_until_ready()
except TypeError as exc:
    raise RuntimeError(
        "JAX runtime failed during a simple array op. Install pinned deps: pip install -r requirements-flax-ml.txt"
    ) from exc

print("Environment ready.")
print("Installed versions:", installed)

In [ ]:
candidate_paths = [
    Path("spectra_tiny.zarr"),
    Path("runs/local-dev/outputs/zarr/synthesized_spectra.zarr"),
]

zarr_path = next((p for p in candidate_paths if p.exists()), None)
if zarr_path is None:
    raise FileNotFoundError("No synthesized spectra store found. Update candidate_paths for your dataset.")

root = zarr.open_group(str(zarr_path), mode="r")
array_names = sorted(root.array_keys())
print("Dataset:", zarr_path)
print("Arrays:", array_names)

if "param_names" not in root or "params" not in root or "flux" not in root:
    raise ValueError("Dataset must contain 'params', 'param_names', and 'flux' arrays.")

param_names = [str(x) for x in np.asarray(root["param_names"][:]).tolist()]
print("param_names:", param_names)

mu_key = None
for candidate in ("mu_selected", "mu"):
    if candidate in root:
        mu_key = candidate
        break
if mu_key is None:
    raise KeyError("Expected a mu array (e.g. 'mu_selected' or 'mu') in the dataset.")

mu_probe = np.asarray(root[mu_key][:], dtype=np.float32)
mu_finite_ratio = float(np.isfinite(mu_probe).mean()) if mu_probe.size else 0.0

print("Using mu array:", mu_key)
print("mu finite ratio:", mu_finite_ratio)
print("rows:", int(root["flux"].shape[0]), "flux_dim:", int(root["flux"].shape[1]))

In [ ]:
batch_size = 16
seed = 7

loaders = create_jax_spectra_dataloaders(
    zarr_path=str(zarr_path),
    batch_size=batch_size,
    input_key="params",
    target_key="flux",
    input_features=param_names,
    train_fraction=0.8,
    val_fraction=0.1,
    normalize_inputs=True,
    normalize_targets=False,
    seed=seed,
)

for split in ("train", "val", "test"):
    loader = loaders[split]
    print(f"{split}: rows={loader.indices.size}, batches={len(loader)}")

In [ ]:
dataset = loaders["train"].dataset

full_flux_dim = dataset.shape("flux")[1]
target_points = 1024
use_log_wavelength = True

if "wavelength" in root:
    wavelength = np.asarray(root["wavelength"][:], dtype=np.float64)
else:
    wavelength = np.arange(full_flux_dim, dtype=np.float64)

if np.any(~np.isfinite(wavelength)):
    raise ValueError("wavelength axis must be finite.")

if use_log_wavelength:
    if np.any(wavelength <= 0.0):
        raise ValueError("wavelength must be strictly positive for log(wavelength) training.")
    source_axis = np.log(wavelength)
    target_axis_label = "log(Wavelength)"
    target_domain = "log-wavelength"
else:
    source_axis = wavelength
    target_axis_label = "Wavelength" if "wavelength" in root else "Pixel index"
    target_domain = "linear wavelength"

if source_axis[0] > source_axis[-1]:
    source_axis = source_axis[::-1]
    reverse_flux = True
else:
    reverse_flux = False

if np.any(np.diff(source_axis) <= 0.0):
    raise ValueError("Source wavelength axis must be strictly monotonic for interpolation.")

target_axis = np.linspace(
    source_axis[0],
    source_axis[-1],
    num=min(target_points, full_flux_dim),
    dtype=np.float64,
)

mu_raw = np.asarray(root[mu_key][:], dtype=np.float32)
mu_source = mu_key

if not np.isfinite(mu_raw).any():
    if "mu_selected_index" in root:
        mu_idx = np.asarray(root["mu_selected_index"][:], dtype=np.float32)
        if np.isfinite(mu_idx).any() and np.any(mu_idx >= 0):
            mu_raw = mu_idx
            mu_source = "mu_selected_index"
        else:
            mu_raw = np.zeros(dataset.row_count, dtype=np.float32)
            mu_source = "constant_zero_fallback"
    else:
        mu_raw = np.zeros(dataset.row_count, dtype=np.float32)
        mu_source = "constant_zero_fallback"

train_ids = np.asarray(loaders["train"].indices, dtype=np.int64)
mu_train = mu_raw[train_ids]
mu_train_finite = np.isfinite(mu_train)
if mu_train_finite.any():
    mu_mean = float(np.mean(mu_train[mu_train_finite]))
    mu_std = float(np.std(mu_train[mu_train_finite]))
else:
    mu_mean = 0.0
    mu_std = 1.0
if mu_std < 1e-6:
    mu_std = 1.0

mu_feature = (np.nan_to_num(mu_raw, nan=mu_mean, posinf=mu_mean, neginf=mu_mean) - mu_mean) / mu_std
mu_feature = mu_feature.astype(np.float32, copy=False)

print("full_flux_dim:", full_flux_dim)
print("target_points:", target_axis.shape[0])
print("target_domain:", target_domain)
print("target axis range:", float(target_axis[0]), float(target_axis[-1]))
print("mu feature source:", mu_source)

if mu_source == "constant_zero_fallback":
    print("Warning: no valid mu values found; using a zero mu feature.")

def batch_to_xy(batch):
    x_params = np.asarray(batch["inputs"], dtype=np.float32)
    if x_params.ndim == 1:
        x_params = x_params[:, None]

    row_ids = np.asarray(batch["indices"], dtype=np.int64)
    mu = mu_feature[row_ids][:, None]

    x = np.concatenate([x_params, mu], axis=1).astype(np.float32, copy=False)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

    y_full = np.asarray(batch["targets"], dtype=np.float32)
    if y_full.ndim == 1:
        y_full = y_full[:, None]

    y = np.empty((y_full.shape[0], target_axis.shape[0]), dtype=np.float32)
    for i in range(y_full.shape[0]):
        src_flux = y_full[i][::-1] if reverse_flux else y_full[i]
        y[i] = np.interp(
            target_axis,
            source_axis,
            src_flux.astype(np.float64, copy=False),
            left=float(src_flux[0]),
            right=float(src_flux[-1]),
        ).astype(np.float32, copy=False)

    y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)
    return x, y

probe_batch = next(iter(loaders["train"]))
x0, y0 = batch_to_xy(probe_batch)
print("x shape (params + mu):", tuple(x0.shape))
print("y shape (flux on selected wavelength grid):", tuple(y0.shape))

In [ ]:
class FluxMLP(nn.Module):
    hidden_dims: tuple[int, ...]
    output_dim: int

    @nn.compact
    def __call__(self, x):
        for width in self.hidden_dims:
            x = nn.Dense(width)(x)
            x = nn.relu(x)
        x = nn.Dense(self.output_dim)(x)
        return x


def create_train_state(rng, model, input_dim, learning_rate):
    variables = model.init(rng, jnp.ones((1, input_dim), dtype=jnp.float32))
    params = variables["params"]
    tx = optax.adam(learning_rate)
    return train_state.TrainState.create(apply_fn=model.apply, params=params, tx=tx)


def loss_components(pred, y, lambda_hi, lambda_lo):
    mse = jnp.mean((pred - y) ** 2)
    hi_pen = jnp.mean(jax.nn.relu(pred - 1.0) ** 2)
    lo_pen = jnp.mean(jax.nn.relu(0.0 - pred) ** 2)
    total = mse + lambda_hi * hi_pen + lambda_lo * lo_pen
    return total, mse, hi_pen, lo_pen


@jax.jit
def train_step(state, x, y, lambda_hi, lambda_lo):
    def loss_fn(params):
        pred = state.apply_fn({"params": params}, x)
        total, mse, hi_pen, lo_pen = loss_components(pred, y, lambda_hi, lambda_lo)
        return total, (mse, hi_pen, lo_pen)

    (loss, (mse, hi_pen, lo_pen)), grads = jax.value_and_grad(loss_fn, has_aux=True)(state.params)
    new_state = state.apply_gradients(grads=grads)
    return new_state, loss, mse, hi_pen, lo_pen


@jax.jit
def eval_step(state, x, y, lambda_hi, lambda_lo):
    pred = state.apply_fn({"params": state.params}, x)
    return loss_components(pred, y, lambda_hi, lambda_lo)


def eval_loader(state, loader, lambda_hi, lambda_lo):
    totals, mses, hi_pens, lo_pens = [], [], [], []
    for batch in loader:
        x_np, y_np = batch_to_xy(batch)
        x = jnp.asarray(x_np, dtype=jnp.float32)
        y = jnp.asarray(y_np, dtype=jnp.float32)
        total, mse, hi_pen, lo_pen = eval_step(state, x, y, lambda_hi, lambda_lo)
        totals.append(float(total))
        mses.append(float(mse))
        hi_pens.append(float(hi_pen))
        lo_pens.append(float(lo_pen))

    if not totals:
        return {"total": float("nan"), "mse": float("nan"), "hi_pen": float("nan"), "lo_pen": float("nan")}

    return {
        "total": float(np.mean(totals)),
        "mse": float(np.mean(mses)),
        "hi_pen": float(np.mean(hi_pens)),
        "lo_pen": float(np.mean(lo_pens)),
    }

In [ ]:
hidden_dims = (128, 256)
learning_rate = 1e-3
num_epochs = 30

# Soft flux-range regularization. Increase lambda_hi if predictions exceed 1 too often.
lambda_hi = 0.1
lambda_lo = 0.0

model = FluxMLP(hidden_dims=hidden_dims, output_dim=y0.shape[1])
state = create_train_state(jax.random.PRNGKey(0), model, x0.shape[1], learning_rate)

history = {
    "train_total": [],
    "val_total": [],
    "train_mse": [],
    "val_mse": [],
    "train_hi_pen": [],
    "val_hi_pen": [],
}

for epoch in range(1, num_epochs + 1):
    train_totals, train_mses, train_hi_pens = [], [], []
    for batch in loaders["train"]:
        x_np, y_np = batch_to_xy(batch)
        x = jnp.asarray(x_np, dtype=jnp.float32)
        y = jnp.asarray(y_np, dtype=jnp.float32)
        state, total, mse, hi_pen, _ = train_step(state, x, y, lambda_hi, lambda_lo)
        train_totals.append(float(total))
        train_mses.append(float(mse))
        train_hi_pens.append(float(hi_pen))

    train_stats = {
        "total": float(np.mean(train_totals)) if train_totals else float("nan"),
        "mse": float(np.mean(train_mses)) if train_mses else float("nan"),
        "hi_pen": float(np.mean(train_hi_pens)) if train_hi_pens else float("nan"),
    }
    val_stats = eval_loader(state, loaders["val"], lambda_hi, lambda_lo)

    history["train_total"].append(train_stats["total"])
    history["val_total"].append(val_stats["total"])
    history["train_mse"].append(train_stats["mse"])
    history["val_mse"].append(val_stats["mse"])
    history["train_hi_pen"].append(train_stats["hi_pen"])
    history["val_hi_pen"].append(val_stats["hi_pen"])

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"epoch={epoch:03d} train_total={train_stats['total']:.6f} val_total={val_stats['total']:.6f} "
            f"train_mse={train_stats['mse']:.6f} val_mse={val_stats['mse']:.6f} "
            f"train_hi_pen={train_stats['hi_pen']:.6f} val_hi_pen={val_stats['hi_pen']:.6f}"
        )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(history["train_mse"], label="train_mse")
axes[0].plot(history["val_mse"], label="val_mse")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE")
axes[0].set_title("Reconstruction error")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history["train_hi_pen"], label="train_hi_pen")
axes[1].plot(history["val_hi_pen"], label="val_hi_pen")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Penalty")
axes[1].set_title("Penalty for flux > 1")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
test_batch = next(iter(loaders["test"]))
x_test, y_test = batch_to_xy(test_batch)
y_pred = np.asarray(state.apply_fn({"params": state.params}, jnp.asarray(x_test, dtype=jnp.float32)))

i = 0
plt.figure(figsize=(10, 4))
plt.plot(target_axis, y_test[i], label="true", lw=1.5)
plt.plot(target_axis, y_pred[i], label="pred", lw=1.2, alpha=0.8)
plt.xlabel(target_axis_label)
plt.ylabel("Flux")
plt.title(f"Held-out spectrum on {target_domain} grid: target vs prediction")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print("Input vector includes", x_test.shape[1], "features (params + mu).")
print("Output points on selected grid:", y_test.shape[1])

## Notes

- Use `use_log_wavelength` in the preprocessing cell to switch between log-wavelength and linear-wavelength targets.
- Targets are built by interpolation onto `target_axis` (size `target_points`) before training.
- The training loss is `MSE + lambda_hi * mean(relu(pred - 1)^2) + lambda_lo * mean(relu(-pred)^2)`.
- Start with `lambda_hi=0.1` and increase it if predicted flux above 1 remains frequent.
- For higher-fidelity training, increase `target_points` and/or model capacity.
- If valid `mu` values are missing in your store, the notebook falls back to a zero-valued `mu` feature and prints a warning.
- Use `requirements-flax-ml.txt` to reproduce the tested FLAX/JAX environment.
- If your dataset stores `mu` under a different key, update `mu_key` detection in the data-inspection cell.